In [173]:
import pandas as pd

df = pd.read_excel("data/Carbon Emission.xlsx")
print(df.head())
print(df.info())

  How Often Shower Heating Energy Source     Transport  \
0            daily                  coal        public   
1  less frequently           natural gas  walk/bicycle   
2  more frequently                  wood       private   
3      twice a day                  wood  walk/bicycle   
4            daily                  coal       private   

  Frequency of Traveling by Air                        Cooking_With  \
0                    frequently                   ['Stove', 'Oven']   
1                        rarely              ['Stove', 'Microwave']   
2                         never               ['Oven', 'Microwave']   
3                        rarely  ['Microwave', 'Grill', 'Airfryer']   
4               very frequently                            ['Oven']   

   Monthly grocery bill  Vehicle Monthly Distance   How Long TV PC Daily   \
0              0.722892                   0.021002               0.291667   
1              0.257028                   0.000900               0.375

In [174]:
y = df["CarbonEmission"]

# Converting existing questionnaire columns into the four desired numeric features

# water_usage: map shower frequency to a numeric scale
shower_map = {
    "never": 0.0,
    "less frequently": 0.5,
    "daily": 1.0,
    "more frequently": 1.5,
    "twice a day": 2.0
}
df["water_usage"] = df["How Often Shower"].map(shower_map).fillna(0)

# transport_km: use vehicle monthly distance directly
df["transport_km"] = df["Vehicle Monthly Distance "]

# electricity_consumption: sum TV/PC and internet usage
df["electricity_consumption"] = (
    df["How Long TV PC Daily "] +
    df["How Long Internet Daily "]
)

# flights_taken: map air travel frequency to a numeric scale
flights_map = {
    "never": 0.0,
    "rarely": 1.0,
    "frequently": 2.0,
    "very frequently": 3.0
}
df["flights_taken"] = df["Frequency of Traveling by Air"].map(flights_map).fillna(0)

# final features
lifestyle_cols = [
    "transport_km",
    "electricity_consumption",
    "water_usage",
    "flights_taken"
]

X = df[lifestyle_cols]

### Using four lifestyle attributes
The model will be trained solely on the following numeric columns:
* `transport_km` (derived from Vehicle Monthly Distance)
* `electricity_consumption` (TV/PC hours plus internet hours)
* `water_usage` (shower frequency mapped: never=0, less frequently=0.5, daily=1, more frequently=1.5, twice a day=2)
* `flights_taken` (air travel frequency mapped: never=0, rarely=1, frequently=2, very frequently=3)

Values are calculated from the original questionnaire columns.

Preprocessing

In [175]:
X.fillna(0, inplace=True)  # fill any missing values with zero

C:\Users\KIIT0001\AppData\Local\Temp\ipykernel_29340\575228066.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X.fillna(0, inplace=True)  # fill any missing values with zero


Train-Test Split

In [176]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [177]:
from sklearn.ensemble import RandomForestRegressor

model = RandomForestRegressor(
    n_estimators=500,
    max_depth=10,
    min_samples_split=2,
    min_samples_leaf=4,
    random_state=42,
    n_jobs=-1
)

model.fit(X_train, y_train)

# remember which columns the model was trained on, required for later predictions
feature_cols = X.columns.tolist()

print(f"Model trained on {len(feature_cols)} features")

Model trained on 4 features


In [178]:
from sklearn.metrics import mean_absolute_error, r2_score

preds = model.predict(X_test)

print("MAE:", mean_absolute_error(y_test, preds))
print("R2:", r2_score(y_test, preds))

MAE: 0.059780664251885114
R2: 0.5552939581013134


In [179]:
import joblib

joblib.dump(model, "models/carbon_emission_model.pkl")

['models/carbon_emission_model.pkl']

Feature Importance

In [180]:
import pandas as pd

importance = pd.Series(
    model.feature_importances_,
    index=X.columns
).sort_values(ascending=False)

print(importance)

transport_km               0.598065
flights_taken              0.324690
electricity_consumption    0.058910
water_usage                0.018334
dtype: float64


In [181]:
# Emission factor for electricity (India grid average)
EMISSION_FACTOR = 0.82  # kg CO2 per kWh


def calculate_energy_carbon(energy_kwh):
    """
    Convert energy consumption to carbon emissions
    """
    return energy_kwh * EMISSION_FACTOR


def calculate_total_carbon(activity_carbon, energy_kwh):
    """
    Combine activity carbon (from preprocessed ML) with energy carbon.
    activity_carbon should already be computed from the model with proper preprocessing.
    """
    energy_carbon = calculate_energy_carbon(energy_kwh)
    total_carbon = activity_carbon + energy_carbon
    
    return {
        "activity_carbon": activity_carbon,
        "energy_carbon": energy_carbon,
        "total_carbon": total_carbon
    }

In [182]:
import pandas as pd

iot_data = pd.read_csv("iot_output.csv")
energy_kwh = iot_data["energy_kwh"].values[0]

print("Energy from IoT model:", energy_kwh)

Energy from IoT model: 3.061892273847997


In [183]:
EMISSION_FACTOR = 0.82

energy_carbon = energy_kwh * EMISSION_FACTOR

print("Carbon from Energy Usage:", energy_carbon, "kg CO2")

Carbon from Energy Usage: 2.5107516645553574 kg CO2


In [184]:
# Build a new observation using the four lifestyle features
new_user = {
    "transport_km": 800,
    "electricity_consumption": 3,
    "water_usage": 2000,
    "flights_taken": 1
}

input_df = pd.DataFrame([new_user])

# fill missing columns just in case
for col in feature_cols:
    if col not in input_df.columns:
        input_df[col] = 0

# match training order (should already match)
input_df = input_df[feature_cols]

activity_carbon = model.predict(input_df)[0]
print("Carbon from Activities:", activity_carbon, "kg CO2")

Carbon from Activities: 0.45327673820075076 kg CO2


In [185]:
final_carbon = energy_carbon + activity_carbon

print("\nFinal Individual Carbon Emission:", final_carbon, "kg CO2")


Final Individual Carbon Emission: 2.9640284027561083 kg CO2


In [186]:
# Example lifestyle features (these would come from user input in a real app)
transport_km = 12
electricity_consumption = 3
water_usage = 2000
flights_taken = 1

In [187]:
def generate_feedback(energy_kwh, transport_km, electricity_consumption, water_usage, flights_taken, final_carbon):

    tips = []

    # IoT Energy usage
    if energy_kwh > 3:
        tips.append(
            "Your household electricity consumption is relatively high. Consider reducing AC usage, turning off idle appliances, and using energy-efficient devices."
        )
    elif energy_kwh > 2:
        tips.append(
            "Your electricity usage is moderate. Using LED lighting and smart power strips can further reduce energy consumption."
        )
    else:
        tips.append(
            "Your electricity consumption is efficient. Maintaining energy-saving habits helps lower carbon emissions."
        )

    # Transport
    if transport_km > 10:
        tips.append(
            "Your daily transport distance is high. Consider carpooling, public transport, cycling, or electric vehicles."
        )
    else:
        tips.append(
            "Your transport usage is relatively efficient. Sustainable commuting helps reduce carbon emissions."
        )

    # Water usage
    if water_usage > 1500:
        tips.append(
            "Your water usage is relatively high. Consider low-flow fixtures, fixing leaks, and reducing unnecessary water consumption."
        )
    else:
        tips.append(
            "Your water consumption is within an efficient range. Continue practicing water conservation."
        )

    # Electricity consumption
    if electricity_consumption > 3:
        tips.append(
            "Electricity usage is high. Try reducing AC usage, switching to LED lights, and using energy-efficient appliances."
        )
    else:
        tips.append(
            "Your electricity usage is within a reasonable range. Using renewable energy sources can further reduce emissions."
        )

    # Flights taken
    if flights_taken >= 3:
        tips.append(
            "Frequent air travel significantly increases carbon emissions. Consider reducing flights or using carbon offset programs."
        )
    elif flights_taken > 0:
        tips.append(
            "Air travel contributes to carbon emissions. If possible, reduce flight frequency or choose sustainable travel options."
        )

    # Overall carbon footprint
    if final_carbon > 6:
        tips.append(
            "Your overall carbon footprint is high. Reducing electricity usage and transport emissions can significantly lower your impact."
        )
    elif final_carbon > 3:
        tips.append(
            "Your carbon footprint is moderate. Small behavioural changes can further reduce your environmental impact."
        )
    else:
        tips.append(
            "Your carbon footprint is relatively low. Continue maintaining sustainable habits."
        )

    return tips

In [188]:
tips = generate_feedback(energy_kwh, transport_km, electricity_consumption, water_usage, flights_taken, final_carbon)

print("\nPersonalised Carbon Reduction Suggestions:")
for t in tips:
    print("-", t)


Personalised Carbon Reduction Suggestions:
- Your household electricity consumption is relatively high. Consider reducing AC usage, turning off idle appliances, and using energy-efficient devices.
- Your daily transport distance is high. Consider carpooling, public transport, cycling, or electric vehicles.
- Your water usage is relatively high. Consider low-flow fixtures, fixing leaks, and reducing unnecessary water consumption.
- Your electricity usage is within a reasonable range. Using renewable energy sources can further reduce emissions.
- Air travel contributes to carbon emissions. If possible, reduce flight frequency or choose sustainable travel options.
- Your carbon footprint is relatively low. Continue maintaining sustainable habits.


# Weekly Carbon Footprint Analysis
Calculating weekly carbon emissions by extrapolating daily measurements over 7 days.

In [190]:
# Calculate weekly carbon emissions (assuming same daily pattern repeats 7 days)
weekly_energy_carbon = energy_carbon * 7
weekly_activity_carbon = activity_carbon * 7
weekly_total_carbon = final_carbon * 7

print("=" * 50)
print("WEEKLY CARBON FOOTPRINT ANALYSIS")
print("=" * 50)
print(f"Daily Activity Carbon:      {activity_carbon:.4f} kg CO2")
print(f"Weekly Activity Carbon:     {weekly_activity_carbon:.4f} kg CO2")
print()
print(f"Daily Energy Carbon:        {energy_carbon:.4f} kg CO2")
print(f"Weekly Energy Carbon:       {weekly_energy_carbon:.4f} kg CO2")
print()
print(f"Daily Total Carbon:         {final_carbon:.4f} kg CO2")
print(f"Weekly Total Carbon:        {weekly_total_carbon:.4f} kg CO2")
print("=" * 50)

WEEKLY CARBON FOOTPRINT ANALYSIS
Daily Activity Carbon:      0.4533 kg CO2
Weekly Activity Carbon:     3.1729 kg CO2

Daily Energy Carbon:        2.5108 kg CO2
Weekly Energy Carbon:       17.5753 kg CO2

Daily Total Carbon:         2.9640 kg CO2
Weekly Total Carbon:        20.7482 kg CO2


In [191]:
def generate_weekly_feedback(weekly_total_carbon, weekly_energy_carbon, weekly_activity_carbon):
    """
    Generate personalized feedback based on weekly carbon emissions.
    Provides recommendations and context for reducing weekly impact.
    """
    weekly_tips = []
    
    # Reference targets (kg CO2 per week)
    # Typical sustainability targets: 20-30 kg CO2/week for a household
    high_threshold = 50      # Very high
    moderate_threshold = 30  # Moderate
    low_threshold = 20       # Low/sustainable
    
    # Overall weekly assessment
    if weekly_total_carbon > high_threshold:
        weekly_tips.append(
            f" HIGH: Your weekly carbon footprint is {weekly_total_carbon:.2f} kg CO2. "
            "This is significantly above sustainable levels. Urgent action needed."
        )
    elif weekly_total_carbon > moderate_threshold:
        weekly_tips.append(
            f" MODERATE: Your weekly carbon footprint is {weekly_total_carbon:.2f} kg CO2. "
            "You're in the moderate range. Small changes can make a real impact."
        )
    else:
        weekly_tips.append(
            f" GOOD: Your weekly carbon footprint is {weekly_total_carbon:.2f} kg CO2. "
            "You're on track for sustainable living!"
        )
    
    # Energy-specific breakdown
    if weekly_energy_carbon > 10:
        weekly_tips.append(
            f"Energy-related emissions ({weekly_energy_carbon:.2f} kg CO2/week) are significant. "
            "Installing renewable energy (solar) or improving insulation can help."
        )
    
    # Activity-specific breakdown
    if weekly_activity_carbon > high_threshold * 0.6:
        weekly_tips.append(
            f"Lifestyle activities ({weekly_activity_carbon:.2f} kg CO2/week) contribute heavily. "
            "Focus on reducing transport frequency, cutting unnecessary flights, and water conservation."
        )
    
    # Cumulative annual impact
    annual_carbon = weekly_total_carbon * 52
    weekly_tips.append(
        f"\n Annual Impact: At this rate, you'll emit ~{annual_carbon:.0f} kg CO2 per year "
        f"({annual_carbon/1000:.1f} tonnes). A sustainable target is ~5–10 tonnes/person/year."
    )
    
    return weekly_tips

In [192]:
# Generate and display weekly feedback
weekly_feedback = generate_weekly_feedback(weekly_total_carbon, weekly_energy_carbon, weekly_activity_carbon)

print("\n" + "=" * 60)
print("PERSONALIZED WEEKLY CARBON REDUCTION STRATEGIES")
print("=" * 60)
for feedback in weekly_feedback:
    print(feedback)
    print()



PERSONALIZED WEEKLY CARBON REDUCTION STRATEGIES
 GOOD: Your weekly carbon footprint is 20.75 kg CO2. You're on track for sustainable living!

Energy-related emissions (17.58 kg CO2/week) are significant. Installing renewable energy (solar) or improving insulation can help.


 Annual Impact: At this rate, you'll emit ~1079 kg CO2 per year (1.1 tonnes). A sustainable target is ~5–10 tonnes/person/year.

